In [1]:
import pathlib
import textwrap

import numpy as np
import pickle
from PIL import Image

from IPython.display import display
from IPython.display import Markdown

# from matplotlib.pyplot import imshow

# from sklearn.metrics import roc_auc_score
import re
import PIL.Image
import json
import os
import pandas as pd


def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

from google import genai

In [2]:
all_files = []
img_dir = """/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/"""
for path, subdirs, files in os.walk(img_dir):
    for name in files:
        all_files.append(os.path.join(path, name))

In [ ]:

client = genai.Client(api_key='XYZ')

In [19]:
print("List of models that support generateContent:\n")
for m in client.models.list():
    for action in m.supported_actions:
        if action == "generateContent":
            print(m.name)

List of models that support generateContent:

models/gemini-1.5-pro-latest
models/gemini-1.5-pro-002
models/gemini-1.5-pro
models/gemini-1.5-flash-latest
models/gemini-1.5-flash
models/gemini-1.5-flash-002
models/gemini-1.5-flash-8b
models/gemini-1.5-flash-8b-001
models/gemini-1.5-flash-8b-latest
models/gemini-2.5-pro-preview-03-25
models/gemini-2.5-flash-preview-05-20
models/gemini-2.5-flash
models/gemini-2.5-flash-lite-preview-06-17
models/gemini-2.5-pro-preview-05-06
models/gemini-2.5-pro-preview-06-05
models/gemini-2.5-pro
models/gemini-2.0-flash-exp
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-exp-image-generation
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.0-flash-preview-image-generation
models/gemini-2.0-flash-lite-preview-02-05
models/gemini-2.0-flash-lite-preview
models/gemini-2.0-pro-exp
models/gemini-2.0-pro-exp-02-05
models/gemini-exp-1206
models/gemini-2.0-flash-thinking-exp-01-21
models/gemini-2.0-flash-thi

In [5]:
safety_settings = [{"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"}, 
                   {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                   {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}]

generation_settings = {"top_p": 0.7, "max_output_tokens": 1024}

In [6]:
all_files[-1]

'/projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Biological/4da96ecd08280d6400e1bd0ea608deea.jpg'

In [7]:
last_processed_idx = -1
unprocessed_ids = []
responses = []

In [8]:
import time

In [36]:
for idx, file in enumerate(all_files):
    if idx <= last_processed_idx:
        continue
    img = PIL.Image.open(file)
    print(f"Processing image: {file}")
    prompt = """I have SEM images that may belong to any of these classes - (1) biological, (2) fibers, (3) coated films, (4) MEMS, (5) nanowires, (6) particles, (7) patterned surfaces, (8) porous sponge, (9) powder, or (10) tips. Can you please identify the image type using one of the above classes? Please answer with the class number only. I want you to make a guess even if you are not sure. I do not want any extra information, only the class number. If it is impossible to determine the class, please answer with `NaN', and nothing else."""
    response = client.models.generate_content(model = 'gemini-2.5-pro', contents = [img, prompt])
    try:
        answer = response.text
    except Exception as e:
        unprocessed_ids.append(path)
        answer = "MODEL ERROR"
    print(answer)
    print()
    responses.append(answer)
    last_processed_idx = idx
    time.sleep(60)

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/fa4e00871cf1efd959217266bf8fb4f0.jpg
4

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/9643ff7135b7de64aa8f58a289748ba5.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/453d3182c8d3e99405e85690a3c15c95.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/10f31f7b4e9182059fc026f92757ed3b.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/db736c42b0eb4dec9405138b3b08144d.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/8d804f30b9e74d341e1f5629fc77032c.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/3c009b897a462c91853dc62d30a4caa1.jpg
10

Processing image: /projects/matsci/vlm_microscopy/Microscopy/NFFA/sampled_data/Tips/4f747b413ee57b511d8c75cb1db85ce1.jpg
10

P

In [28]:
# f8365919f611dcc61903c5ef7093ba6f

In [39]:
len(responses)

250

In [40]:
response.candidates

[Candidate(
   content=Content(
     parts=[
       Part(
         text='1'
       ),
     ],
     role='model'
   ),
   finish_reason=<FinishReason.STOP: 'STOP'>,
   index=0
 )]

In [41]:
response.prompt_feedback

In [42]:
len(unprocessed_ids)

0

In [43]:
with open('results_nffa_full_10classes_prediction_gemini.pkl', 'wb') as f:
    pickle.dump({"responses": responses, "unprocessed_ids": unprocessed_ids}, f)

In [44]:
with open('results_nffa_full_10classes_prediction_gemini.pkl', 'rb') as f:
    data = pickle.load(f)
    responses = data['responses']

In [45]:
counts = [(elem, sum(np.array(responses) == elem)) for elem in np.unique(np.array(responses))]

In [46]:
counts

[(np.str_('1'), np.int64(22)),
 (np.str_('10'), np.int64(22)),
 (np.str_('2'), np.int64(25)),
 (np.str_('3'), np.int64(32)),
 (np.str_('4'), np.int64(15)),
 (np.str_('5'), np.int64(38)),
 (np.str_('6'), np.int64(27)),
 (np.str_('7'), np.int64(23)),
 (np.str_('8'), np.int64(31)),
 (np.str_('9'), np.int64(14)),
 (np.str_('NaN'), np.int64(1))]

In [47]:
len(unprocessed_ids)

0

In [48]:
unprocessed_ids

[]

In [56]:
folder_id_map = {"Biological": 1,
                "Fibres": 2,
                "Films_Coated_Surface": 3,
                "MEMS_devices_and_electrodes": 4,
                "Nanowires": 5,
                "Particles": 6,
                "Patterned_surface": 7,
                "Porous_Sponge": 8,
                "Powder": 9,
                "Tips": 10}

class_id_map = {"biological": 1,
               "fibers": 2,
               "coated film": 3,
               "mems": 4,
               "nanowire": 5,
               "particles": 6,
               "patterned surface": 7,
               "porous sponge": 8,
               "powder": 9,
               "tips": 10}

actuals = []
predictions = []

import re

predictions = []
for idx, path in enumerate(all_files):
    numbers = re.findall(r'\d+', responses[idx])
    pred = -1
    if len(numbers) > 0:
        pred = int(numbers[0])
    predictions.append(pred)

for idx, file in enumerate(all_files):
    extracted_folder = file.split("/")[-2]
    actual_class_id = folder_id_map[extracted_folder]
    actuals.append(actual_class_id)

In [57]:
len(actuals)

250

In [58]:
len(predictions)

250

In [59]:
actuals[:10]

[5, 5, 5, 5, 5, 5, 5, 5, 5, 5]

In [60]:
predictions[:10]

[5, 5, 5, 5, 5, 5, 3, 5, 5, 5]

In [61]:
accuracy = (np.array(actuals) == np.array(predictions)).sum()/len(actuals)

In [62]:
accuracy

np.float64(0.764)

In [63]:
np.where(np.array(predictions) == -1)

(array([75]),)

In [64]:
responses[41]

'7'

In [68]:
import pandas as pd
df = pd.DataFrame({'img_path': all_files, 'raw_predictions': responses, 'actuals': actuals, 'predictions': predictions})

In [69]:
df.to_csv('classification_NFFA_gemini_sampled_data.csv')